In [1]:
import os
import json

import numpy as np
import pandas as pd

In [2]:
runs = [
    os.path.join('experiments', str(x)) for x in range(1657, 1670)
]

assert len(runs) == 13, len(runs)

In [3]:
soln_number_to_name = {
    1: 'Plane waves',
    2: 'Radial waves',
    3: 'Hopf fibration',
    4: 'Random solution'
}

In [4]:
df = pd.DataFrame(
    columns=list(json.load(open(os.path.join(runs[0], 'hparams.json'))).keys())
    + [
        'soln_name',
        'min_mse_val',
        'time_to_min_mse_val',
        'min_rel_l2_error',
        'time_to_min_rel_l2_error',
        'time_to_5p_rel_l2_error'
    ]
)

for run in runs:
    info = json.load(open(os.path.join(run, 'hparams.json')))

    assert (info['do_masking'] and info['add_bc']) or (not info['add_bc'])
    
    info['soln_name'] = soln_number_to_name[info['soln']]
    
    log = pd.read_csv(os.path.join(run, 'log.tsv'), sep='\t')
    
    # print(log['training_time'].cumsum().max())
    
    # epoch where we reach minimum validation MSE
    min_mse_val = log['mse_val'].min()
    info['min_mse_val'] = min_mse_val
    
    min_mse_val_epoch = int(log[log['mse_val'] == min_mse_val].iloc[0]['epoch'])
    # how much training time to get there?
    info['time_to_min_mse_val'] = log[log['epoch'] <= min_mse_val_epoch]['training_time'].sum().item()
    
    # epoch where we reach minimum validation relative L2 error
    min_rel_l2_error = log['rel_l2_error'].min()
    info['min_rel_l2_error'] = min_rel_l2_error
    
    min_rel_l2_error_epoch = int(log[log['rel_l2_error'] <= min_rel_l2_error].iloc[0]['epoch'])
    info['time_to_min_rel_l2_error'] = log[log['epoch'] <= min_rel_l2_error_epoch]['training_time'].sum().item()
    
    # epoch where we first reach 5% relative error
    first_5p_epoch = int(log[log['rel_l2_error'] < 0.05].iloc[0]['epoch'])
    info['time_to_5p_rel_l2_error'] = log[log['epoch'] <= first_5p_epoch]['training_time'].sum().item()
    
    df.loc[len(df)] = info

In [5]:
df['min_rel_l2_error'] = df['min_rel_l2_error'] * 100

df = df.round(3)

for col in df.columns:
    df[col] = df[col].astype(str)

In [6]:
results = df[[
    'soln_name',
    'width',
    'activation',
    # 'n_train',
    'time_to_5p_rel_l2_error',
    'min_rel_l2_error',
    'time_to_min_rel_l2_error',
]].rename(columns={
    'time_to_5p_rel_l2_error': 'Time to <5% RL2E. (s)',
    'time_to_min_rel_l2_error': 'Time to min. RL2E. (s)',
    'min_rel_l2_error': 'Min. RL2E. (%)',
    'soln_name': 'Solution',
    'width': 'Network width',
    'activation': 'Activation'
    # 'n_train': '# points'
})

results

,Solution,Network width,Activation,Time to <5% RL2E. (s),Min. RL2E. (%),Time to min. RL2E. (s)
0,Random solution,100,tanh,488.181,4.947,488.181
1,Random solution,200,tanh,296.99,3.222,501.187
2,Random solution,500,tanh,202.835,3.132,506.354
3,Random solution,1000,tanh,126.329,2.558,495.078
4,Random solution,2000,tanh,98.796,2.052,475.182
5,Random solution,5000,tanh,78.851,2.048,492.17
6,Random solution,10000,tanh,53.94,1.589,499.322
7,Random solution,12000,tanh,50.437,1.589,474.727
8,Random solution,10000,relu,326.11,3.879,503.41
9,Random solution,10000,silu,118.156,2.189,496.715


In [7]:
print(results.to_latex())

\begin{tabular}{lllllll}
\toprule
{} &         Solution & Network width & Activation & Time to <5\% RL2E. (s) & Min. RL2E. (\%) & Time to min. RL2E. (s) \\
\midrule
0  &  Random solution &           100 &       tanh &               488.181 &          4.947 &                488.181 \\
1  &  Random solution &           200 &       tanh &                296.99 &          3.222 &                501.187 \\
2  &  Random solution &           500 &       tanh &               202.835 &          3.132 &                506.354 \\
3  &  Random solution &          1000 &       tanh &               126.329 &          2.558 &                495.078 \\
4  &  Random solution &          2000 &       tanh &                98.796 &          2.052 &                475.182 \\
5  &  Random solution &          5000 &       tanh &                78.851 &          2.048 &                 492.17 \\
6  &  Random solution &         10000 &       tanh &                 53.94 &          1.589 &                499.32

/var/folders/54/ctkh4lg97kd4x1fv3qzrpd3c0000gp/T/ipykernel_45729/3863709051.py:1: FutureWarning: In future versions `DataFrame.to_latex` is expected to utilise the base implementation of `Styler.to_latex` for formatting and rendering. The arguments signature may therefore change. It is recommended instead to use `DataFrame.style.to_latex` which also contains additional functionality.
  print(results.to_latex())
